# Audiovisual Integration of stimuli for stress perception experiment
This script conists of two parts. In the first section, the optimal duration of a videoclipo is identified in order to integrate it with a list of audioclips. In the second section, the audio and video files are integrated.

# Find optimal duration of viudeoclip and gesture timing in order to integrate it with different audioclips

In [ ]:
# Find the best moment for the beat gesture in the videos
# and the best video duration.
#
# FYI: The beat gesture occurs at the same moment as the vowel onset
# in the audio. Every audio clip his padded with 50 ms of white noise at start and end,
# If one of these is negative, the audio does not fit video

from pathlib import Path
import csv
import wave
import numpy as np

# Settings and initizialation of variables

# folder with the audio files for experimental conditions
audio_dir = Path("/content/Audio_exp/")
# script should be run seperately for distractors because they all got rejected when including with the stimuli
audio_dir_distractors = Path("/content/Audio_exp/Distractors")

# file with the vowel onsets (made in Praat)
timings_file = audio_dir / "splitaudio_annotations_praatlog.txt"

# video duration in seconds
# I wanted to see whether a video of 6 s would work for all conditions
min_video_dur = 6
max_video_dur = 6

# earliest and latest moment for the beat gesture in the video (seconds)
min_t_beat = 0.50
max_t_beat = 5.00

# the beat time we use at the moment, used when two options are equally good
current_beat = 2.0

# step size when trying different values (seconds)
timestep = 0.1

# Step 1: read the audio files and the timings

# create empty list to store clips
clips = []

with open(timings_file, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")

    # go through the file row by row (one row = one audio clip)
    for row in reader:
        filename = row["audiofile"].strip()
        wav_path = audio_dir / filename

        # skip the file if it does not exist
        if not wav_path.exists():
            print("Missing:", filename)
            continue

        # calculate how long the clip is (number of frames / frames per second)
        with wave.open(str(wav_path), "rb") as wav:
            duration = wav.getnframes() / wav.getframerate()

        # save everything about this clip in a dictionary
        clip = {
            "file": filename,
            "duration": duration,
            "t.strong1": float(row["t.strong1"]),
            "t.strong2": float(row["t.strong2"]),
            "t.weak1": float(row["t.weak1"]),
            "t.weak2": float(row["t.weak2"]),
        }
        clips.append(clip)

print("Number of clips found:", len(clips))

# Step 2: make lists of all values we want to try

# all beat times from min to max
# (+ timestep / 2 makes sure the last value is also included)
beats = np.round(np.arange(min_t_beat, max_t_beat + timestep / 2, timestep), 2)

# all video durations from min to max
durations = np.round(np.arange(min_video_dur, max_video_dur + timestep / 2, timestep), 2)

# the 4 conditions: a name and the column name in the Praat file
conditions = [
    ("tStrong1", "t.strong1"),
    ("tStrong2", "t.strong2"),
    ("tWeak1", "t.weak1"),
    ("tWeak2", "t.weak2"),
]

# Step 3: try everything for each condition

for condition_name, onset_column in conditions:

    # variables to remember the best option so far
    best_beat = None
    best_duration = None
    best_accepted = []
    best_rejected = []

    # try every beat time
    for beat in beats:

        # try every video duration
        for video_duration in durations:

            # lists for clips that fit and clips that do not fit
            accepted = []
            rejected = []

            # check every clip
            for clip in clips:
                onset = clip[onset_column]
                audio_duration = clip["duration"]

                # silence needed before and after the audio
                pad_begin = beat - onset
                pad_end = video_duration - audio_duration - pad_begin

                # the clip fits if there is no negative silence
                # (-0.000000001 instead of 0 because of small rounding errors)
                if pad_begin >= -0.000000001 and pad_end >= -0.000000001:
                    accepted.append(clip["file"])
                else:
                    rejected.append(clip["file"])

            # now decide if this option is better than the best one so far
            is_better = False

            if best_beat is None:
                # first option we try, so it is the best for now
                is_better = True
            elif len(accepted) > len(best_accepted):
                # 1. more clips fit
                is_better = True
            elif len(accepted) == len(best_accepted):
                if video_duration < best_duration:
                    # 2. same number of clips, but shorter video
                    is_better = True
                elif video_duration == best_duration:
                    # 3. same video duration, take the beat closest to current beat
                    if abs(beat - current_beat) < abs(best_beat - current_beat):
                        is_better = True

            # if it is better, save it
            if is_better:
                best_beat = beat
                best_duration = video_duration
                best_accepted = accepted
                best_rejected = rejected

    # Step 4: print the result for this condition
    print()
    print("Condition:", condition_name)
    print("Best beat time:", best_beat, "s")
    print("Best video duration:", best_duration, "s")
    print("Accepted:", len(best_accepted), "out of", len(clips))
    print("Rejected:", len(best_rejected), "out of", len(clips))

    # only print the list of rejected files if there are any
    if len(best_rejected) > 0:
        print("Rejected files:")
        for filename in best_rejected:
            print("  ", filename)

# Integrate audio and video files using ffmpeg

In [ ]:
from pathlib import Path
import ffmpeg

# Create a list with all audio files
audio_dir = Path("/content/Integration_files/")
audios = sorted(audio_dir.rglob("*.wav"))

print(f"In total there are {len(audio_files)} WAV files.")

# List the video files corresponding to our gesture conditions and the corresponding output directories to save integrated videos to
videos = [
    (
        "/content/Integration_files/Grooming_gesture2integrate.mp4",
        Path("/content/Integration_files/Output_grooming"),
    ),
    (
        "/content/Integration_files/Beat_gesture2integrate.mp4",
        Path("/content/Integration_files/Output_beat"),
    ),
]

# Keep track of videos which cannot be integrated
fails = []

# Loop through the two videos
for video_file, output_dir in videos:

  # Check to make sure output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)

    # Loop through audio files
    for audio_file in audio_files:
      # Ensure output files are named right
        output_name =
        output_path = output_dir / f"{output_name}.mp4"

        try:
            video = ffmpeg.input(video_file)
            audio = ffmpeg.input(str(audio_file))

            (
                ffmpeg
                .output(
                    video.video,
                    audio.audio,
                    str(output_path),
                    vcodec="copy",
                    acodec="aac",
                    shortest=None,
                )
                .overwrite_output()
                .run(capture_stdout=True, capture_stderr=True)
            )

            print(f"Created: {output_path.name}")

        except Exception as e:
            print(f"\nFAILED")
            print(f"Video: {video_file}")
            print(f"Audio: {audio_file}")
            print(f"Error: {type(e).__name__}: {e}")

            if isinstance(e, ffmpeg.Error) and e.stderr:
                print(e.stderr.decode("utf-8", errors="replace"))

            failed_files.append(
                (video_file, str(audio_file), str(e))
            )

print("\n========== SUMMARY ==========")
print(f"Successful outputs: {2 * len(audio_files) - len(failed_files)}")
print(f"Failed outputs: {len(failed_files)}")

for video_file, audio_file, error in failed_files:
    print(f"Video: {Path(video_file).name} | Audio: {Path(audio_file).name}")
    print(f"Error: {error}\n")

Found 496 WAV files.
Created: A10_1__trimmed_padded.mp4
Created: A10_2__trimmed_padded.mp4
Created: A11_1__trimmed_padded.mp4
Created: A11_2__trimmed_padded.mp4
Created: A12_1__trimmed_padded.mp4
Created: A12_2__trimmed_padded.mp4
Created: A13_1__trimmed_padded.mp4
Created: A13_2__trimmed_padded.mp4
Created: A14_1__trimmed_padded.mp4
Created: A14_2__trimmed_padded.mp4
Created: A15_1__trimmed_padded.mp4
Created: A15_2__trimmed_padded.mp4
Created: A16_1__trimmed_padded.mp4
Created: A16_2__trimmed_padded.mp4
Created: A17_1__trimmed_padded.mp4
Created: A17_2__trimmed_padded.mp4
Created: A18_2__trimmed_padded.mp4
Created: A19_1_trimmed_padded.mp4
Created: A19_2__trimmed_padded.mp4
Created: A1_1__trimmed_padded.mp4
Created: A1_2__trimmed_padded.mp4
Created: A20_1__trimmed_padded.mp4
Created: A20_2__trimmed_padded.mp4
Created: A21_1__trimmed_padded.mp4
Created: A21_2__trimmed_padded.mp4
Created: A22_1__trimmed_padded.mp4
Created: A22_2__trimmed_padded.mp4
Created: A23_1__trimmed_padded.mp4
Cr

KeyboardInterrupt: 